# RK1828 RAFT-Stereo Disparity Demo


[RAFT-Stereo](https://github.com/princeton-vl/RAFT-Stereo) (Princeton Vision &
Learning Lab) estimates a **disparity map** from a stereo pair: brighter means
closer. This is the realtime variant, 160x240 input, int8 quantized, about
90 ms/pair on the RK1828.

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Run and view the disparity map |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("raft-stereo-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/raft_stereo/demo_x86_64.tar.gz"),
    ("raft-stereo-rknn3-rk1828", "model/raftstereo-realtime-160x240.rknn", "demos/raft_stereo/model/raftstereo-realtime-160x240.rknn"),
    ("raft-stereo-rknn3-rk1828", "model/raftstereo-realtime-160x240.weight", "demos/raft_stereo/model/raftstereo-realtime-160x240.weight"),
    ("raft-stereo-rknn3-rk1828", "model/xt_left.jpg", "demos/raft_stereo/model/xt_left.jpg"),
    ("raft-stereo-rknn3-rk1828", "model/xt_right.jpg", "demos/raft_stereo/model/xt_right.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/raft_stereo/demo_x86_64.tar.gz", "demos/raft_stereo")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Run the demo

Input: `model/xt_left.jpg` / `model/xt_right.jpg` (a sample stereo pair)
Output: `out.png` disparity heatmap

In [ ]:
show_images(["demos/raft_stereo/model/xt_left.jpg",
             "demos/raft_stereo/model/xt_right.jpg"])

In [ ]:
DEMO = "demos/raft_stereo"
before = snapshot(DEMO)
run_demo(["./rknn_raft_stereo_demo",
          "model/raftstereo-realtime-160x240.rknn",
          "model/raftstereo-realtime-160x240.weight",
          "model/xt_left.jpg", "model/xt_right.jpg", "0x1"], DEMO)
show_new_images(DEMO, before)

## Summary

- Disparity = horizontal shift of the same point between left and right
  images; larger disparity means closer. It is the core of stereo depth.
- Try your own stereo photos (keep 160x240, or let the demo scale them).